In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import TargetEncoder
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e2/train.csv
/kaggle/input/competitions/playground-series-s6e2/test.csv


In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')
train['source'] = 'train'
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv')

NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

for df in [train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')

FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index', 'strat_feature']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X = X.fillna(0)
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET]

X_test = test[FEATURES]
# X_test = X_test.fillna(0)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
13


In [3]:
BASE_FEATURES = ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

for c in BASE_FEATURES:
    n = f'{c}_mean_te'
    TE = TargetEncoder(cv=5, random_state=42, shuffle=True)
    X[n] = TE.fit_transform(pd.DataFrame(X[c]), y).flatten()
    X_test[n] = TE.transform(pd.DataFrame(X_test[c])).flatten()

/tmp/ipykernel_17/864582437.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[n] = TE.fit_transform(pd.DataFrame(X[c]), y).flatten()
/tmp/ipykernel_17/864582437.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[n] = TE.transform(pd.DataFrame(X_test[c])).flatten()
/tmp/ipykernel_17/864582437.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

In [4]:
CATS = []
for col in NUMS:
    n = f'{col}_cat'
    for df in [X, X_test]:
        df[n] = df[col].astype(str).astype('category')
    CATS.append(n)

In [5]:
import h2o
from h2o.automl import H2OAutoML

# Initialize H2O
h2o.init()

# Load your training data into H2OFrame
# y = y.astype('category')
h2o_df = h2o.H2OFrame(pd.concat([X, y], axis=1))
h2o_df[CONFIG.TARGET] = h2o_df[CONFIG.TARGET].asfactor()
features = X.columns.to_list()

aml = H2OAutoML(max_models=35,
                max_runtime_secs=42000,
                seed=42,
                sort_metric='AUC',
                nfolds=5,
                # balance_classes=True,
                keep_cross_validation_predictions=True
                # include_algos=["GBM", "XGBoost", "StackedEnsemble", "GLM", "DRF", "deeplearning"],
#                 algo_parameters=hyperparameters,
               )

# Train the AutoML model
aml.train(x=features, y=CONFIG.TARGET, training_frame=h2o_df)

# Display the leaderboard
lb = aml.leaderboard
print(lb)


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.17" 2025-10-21; OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04); OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmps02nitne
  JVM stdout: /tmp/tmps02nitne/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmps02nitne/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,04 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,3 months
H2O_cluster_name:,H2O_from_python_unknownUser_x68xr7
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.500 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%
model_id                                                      auc    logloss     aucpr    mean_per_class_error      rmse        mse
StackedEnsemble_AllModels_1_AutoML_1_20260224_210946     0.955549   0.267424  0.948898                0.112039  0.284613  0.0810044
StackedEnsemble_BestOfFamily_1_AutoML_1_20260224_210946  0.955503   0.267608  0.948832                0.112086  0.284664  0.0810338
GBM_5_AutoML_1_20260224_210946                           0.955324   0.268979  0.948585                0.11233   0.285178  0.0813267
XGBoost_grid_1_AutoML_1_20260224_210946_model_4          0.955269   0.268275  0.948551                0.112596  0.2851    0.0812818
GBM_2_AutoML_1_20260224_210946                           0.955256   0.269106  0.948504                0.112304  0.285295  0.0813934
GBM_grid_1_AutoML_

In [6]:
# import h2o
# from h2o.automl import H2OAutoML

# h2o.init()
# h2o_df = h2o.H2OFrame(training_data)

# h2o_df['loan_status'] = h2o_df['loan_status'].asfactor()

# y = 'loan_status'
# X = [col for col in cols if col!='loan_status']

# aml = H2OAutoML(max_models=30, max_runtime_secs=39600, seed=42,
#                 sort_metric='AUC', nfolds=5,
#                 balance_classes=True, include_algos=["GBM", "XGBoost", "StackedEnsemble", "GLM", "DRF"])
# aml.train(x=X, y=y, training_frame=h2o_df)


In [7]:
lb = aml.leaderboard
lb.head(20)

model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
StackedEnsemble_AllModels_1_AutoML_1_20260224_210946,0.955549,0.267424,0.948898,0.112039,0.284613,0.0810044
StackedEnsemble_BestOfFamily_1_AutoML_1_20260224_210946,0.955503,0.267608,0.948832,0.112086,0.284664,0.0810338
GBM_5_AutoML_1_20260224_210946,0.955324,0.268979,0.948585,0.11233,0.285178,0.0813267
XGBoost_grid_1_AutoML_1_20260224_210946_model_4,0.955269,0.268275,0.948551,0.112596,0.2851,0.0812818
GBM_2_AutoML_1_20260224_210946,0.955256,0.269106,0.948504,0.112304,0.285295,0.0813934
GBM_grid_1_AutoML_1_20260224_210946_model_1,0.955184,0.270131,0.94842,0.112414,0.28537,0.081436
XGBoost_3_AutoML_1_20260224_210946,0.955108,0.268861,0.948281,0.112299,0.285351,0.0814251
GBM_3_AutoML_1_20260224_210946,0.955106,0.269512,0.948309,0.112527,0.285533,0.0815288
GBM_grid_1_AutoML_1_20260224_210946_model_6,0.955041,0.270219,0.948239,0.112668,0.285565,0.0815475
GBM_grid_1_AutoML_1_20260224_210946_model_5,0.954936,0.269862,0.948087,0.112849,0.285751,0.0816538


In [8]:
model_save_dir = '/kaggle/working/raw_data_models_final/'
os.makedirs(model_save_dir, exist_ok=True)

# 1. Initialize storage for OOF and Test predictions
# We use a dictionary to collect columns before making a DataFrame
oof_df_all = pd.DataFrame()
test_predictions_all = pd.DataFrame()

leaderboard = aml.leaderboard
model_ids = leaderboard['model_id'].as_data_frame()['model_id'].tolist()

print(f"Processing {len(model_ids)} models...")

for model_id in model_ids:
    model = h2o.get_model(model_id)

    h2o.save_model(model=model, path=model_save_dir, force=True)
    try:
        oof_h2o = model.cross_validation_holdout_predictions()
        oof_df_all[model_id] = oof_h2o['p1'].as_data_frame().iloc[:, 0]
    except Exception as e:
        print(f"Could not get OOF for {model_id}: {e}")

    if 'test_h2o' in locals() or 'test_h2o' in globals():
        test_pred_h2o = model.predict(test_h2o)
        test_predictions_all[model_id] = test_pred_h2o['p1'].as_data_frame().iloc[:, 0]

oof_df_all.to_csv(os.path.join(model_save_dir, 'oof_predictions_automl.csv'), index=False)
test_predictions_all.to_csv(os.path.join(model_save_dir, 'test_inference_automl.csv'), index=False)

leaderboard_df = leaderboard.as_data_frame()
leaderboard_df.to_csv(os.path.join(model_save_dir, 'leaderboard.csv'), index=False)

print("All models, OOF, and Test predictions saved successfully.")

# h2o.shutdown(prompt=False)

/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Processing 37 models...
Could not get OOF for StackedEnsemble_AllModels_1_AutoML_1_20260224_210946: 'NoneType' object is not subscriptable
Could not get OOF for StackedEnsemble_BestOfFamily_1_AutoML_1_20260224_210946: 'NoneType' object is not subscriptable


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow a

All models, OOF, and Test predictions saved successfully.


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [9]:
oof_df_all

,GBM_5_AutoML_1_20260224_210946,XGBoost_grid_1_AutoML_1_20260224_210946_model_4,GBM_2_AutoML_1_20260224_210946,GBM_grid_1_AutoML_1_20260224_210946_model_1,XGBoost_3_AutoML_1_20260224_210946,GBM_3_AutoML_1_20260224_210946,GBM_grid_1_AutoML_1_20260224_210946_model_6,GBM_grid_1_AutoML_1_20260224_210946_model_5,GLM_1_AutoML_1_20260224_210946,XGBoost_grid_1_AutoML_1_20260224_210946_model_8,...,DeepLearning_grid_1_AutoML_1_20260224_210946_model_3,XGBoost_grid_1_AutoML_1_20260224_210946_model_7,XGBoost_grid_1_AutoML_1_20260224_210946_model_3,GBM_grid_1_AutoML_1_20260224_210946_model_3,DeepLearning_grid_2_AutoML_1_20260224_210946_model_3,XRT_1_AutoML_1_20260224_210946,GBM_grid_1_AutoML_1_20260224_210946_model_4,XGBoost_grid_1_AutoML_1_20260224_210946_model_1,XGBoost_1_AutoML_1_20260224_210946,DRF_1_AutoML_1_20260224_210946
0,0.992878,0.997166,0.993005,0.988162,0.994409,0.993294,0.989646,0.993213,0.997304,0.995795,...,0.999968,0.998809,0.998373,0.991740,0.999918,0.998591,0.990298,0.999000,0.997500,1.000000
1,0.016553,0.010174,0.017475,0.019942,0.010286,0.016535,0.020177,0.014639,0.009152,0.015784,...,0.007401,0.012444,0.011185,0.014299,0.001520,0.008217,0.021702,0.011995,0.012590,0.031997
2,0.013205,0.011007,0.012505,0.018924,0.012405,0.013244,0.015351,0.012653,0.008787,0.007183,...,0.002199,0.001675,0.002863,0.016072,0.001051,0.018757,0.010469,0.001095,0.003201,0.001841
3,0.030647,0.030655,0.033147,0.043786,0.040888,0.037394,0.034754,0.037809,0.063102,0.045222,...,0.017353,0.070435,0.072966,0.072706,0.014012,0.073538,0.033966,0.053296,0.056361,0.022184
4,0.991073,0.997986,0.990124,0.984221,0.997223,0.989595,0.986721,0.990305,0.997882,0.996594,...,0.999999,0.999365,0.997723,0.990244,0.999943,0.999962,0.991683,0.998777,0.999265,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,0.187537,0.154058,0.190990,0.202968,0.199605,0.164770,0.184205,0.207388,0.166924,0.165126,...,0.072645,0.203547,0.282454,0.159753,0.248877,0.317520,0.117752,0.068956,0.265828,0.211172
629996,0.525921,0.568189,0.537521,0.604762,0.519456,0.549338,0.534456,0.554371,0.525153,0.494869,...,0.699188,0.577921,0.498220,0.524060,0.432331,0.512176,0.469822,0.447172,0.373588,0.409343
629997,0.869451,0.767598,0.862395,0.856251,0.873389,0.864520,0.901760,0.883535,0.834633,0.864302,...,0.953597,0.823656,0.802115,0.879304,0.953892,0.862126,0.892099,0.931239,0.872440,0.720265
629998,0.540161,0.494499,0.491810,0.479671,0.483185,0.511064,0.483834,0.460380,0.465313,0.489344,...,0.395950,0.382435,0.463632,0.472210,0.607432,0.498464,0.460880,0.491530,0.501942,0.608358


In [10]:
# lb = pd.read_csv('/kaggle/working/raw_data_models/leaderboard.csv')
# lb.head(20)

In [11]:
# model_path = h2o.save_model(model=aml.leader, path="./best_model", force=True)

# print(f"Model saved at: {model_path}")

In [12]:
# model = h2o.load_model('/kaggle/working/best_model/StackedEnsemble_AllModels_1_AutoML_6_20241013_182949')
# # model.predict()

In [13]:
# lb = aml.leaderboard

# lb_df = lb.as_data_frame()

# for model_id in lb_df['model_id']:
#     model = h2o.get_model(model_id) 
#     model_path = h2o.save_model(model, path="./models", force=True) 
#     print(f"Model {model_id} saved to {model_path}")

In [14]:
# lb_df

In [15]:
# test_df = pd.read_csv('/kaggle/input/playground-series-s4e10/test.csv')
# idx = test['id']
# test.drop(columns='id', inplace=True)

In [16]:
# h2o_test = h2o.H2OFrame(test_df)
# # predictions_dict = {}
# # prediction_df = pd.DataFrame()

# # # Loop through the first five models from the leaderboard
# # for i in range(5):
# #     model_id = lb_df['model_id'][i]  # Get the model ID
# #     model = h2o.get_model(model_id)  # Retrieve the model using its ID
    
# #     # Make predictions on the test data
# #     predictions = model.predict(h2o_test)
    
# #     # Convert predictions to pandas dataframe
# #     predictions_df = predictions.as_data_frame()
    
# #     # Check the number of columns in predictions_df
# #     print(f"Model ID: {model_id}, Predictions shape: {predictions_df.shape}")

# #     # If predictions_df has more than one column, select the first column (typically the probabilities)
# #     if predictions_df.shape[1] > 1:
# #         predictions_df = predictions_df.iloc[:, 0]  # Select the first column (probabilities)
    
# #     # Rename the column to the model ID
# #     predictions_df = predictions_df.to_frame(name=model_id)  # Convert back to DataFrame with model_id as the column name
    
# #     # Store in dictionary
# #     predictions_dict[model_id] = predictions_df
    
# #     # Concatenate predictions to prediction_df
# #     if prediction_df.empty:
# #         prediction_df = predictions_df  # If prediction_df is empty, initialize it
# #     else:
# #         prediction_df = pd.concat([prediction_df, predictions_df], axis=1)  # Concatenate along the columns
    
# #     # Optional: Print model and first few rows of predictions
# #     print(predictions_df.head())

# # # After the loop, you can check the combined predictions DataFrame
# # print("Combined Predictions DataFrame:")
# # print(prediction_df.head())

In [17]:
# preds = model.predict(h2o_test)[:,2]

In [18]:
# preds_pandas = preds.as_data_frame()

In [19]:
# test = pd.read_csv('/kaggle/input/playground-series-s4e10/test.csv')
# submission_df = pd.DataFrame({
#     'id':test['id'],
#     'loan_status':preds_pandas['p1'].values
# })

In [20]:
# submission_df.to_csv('submission.csv', index=False)

In [21]:
# leaderboard = aml.leaderboard

# # Specify a directory to save the models
# model_save_dir = '/path/to/save/raw_data_models/'  # Change this to your desired directory

# # Loop through the leaderboard and save each model
# for model_id in leaderboard['model_id'].as_data_frame()['model_id']:
#     model = h2o.get_model(model_id)
#     model_path = h2o.save_model(model=model, path=model_save_dir, force=True)
#     print(f"Model {model_id} saved to {model_path}")

# # # Shutdown H2O (optional, if you're done with the session)
# # h2o.shutdown()